# 01 – Data Factory: Q&A Dataset Generation

**Operation Ledger-Mind: The Financial Intelligence**

**Goal:** Transform Uber's 2024 Annual Report PDF into structured instruction dataset.

**Pipeline:**
1. Load and clean PDF
2. Chunk into 1500-character segments
3. Generate 10 Q&A pairs per chunk using dual-LLM pipeline
4. Split into train (80%) and test (20%) sets

In [1]:
# Setup: Config & Services
import sys
import os
from pathlib import Path
from dotenv import load_dotenv
import json
from tqdm import tqdm

# Add parent directory to path
sys.path.append(str(Path.cwd().parent))

# Load environment
load_dotenv()

# Import from src
from src.services.llm_services import (
    load_config,
    get_llm,
    validate_api_keys,
    print_config_summary
)

# Load config
config = load_config("../src/config/config.yaml")
validate_api_keys(config, verbose=True)
print_config_summary(config)

✅ Config loaded:
  LLM: groq / llama-3.1-8b-instant
  Embeddings: sbert / sentence-transformers/all-MiniLM-L6-v2
  Temperature: 0.2
  Artifacts: ./artifacts


In [5]:
# Initialize LLM
llm = get_llm(config)
print(f"✅ LLM initialized")

# Test connection
print("\nTesting API connection...")
try:
    test_response = llm.invoke("Say 'API working!' if you can read this.")
    test_msg = test_response.content if hasattr(test_response, 'content') else str(test_response)
    print(f"✅ API verified: {test_msg[:50]}")
except Exception as e:
    print(f"❌ API test failed: {e}")

✅ LLM initialized

Testing API connection...
✅ API verified: API working!


## Step 1: PDF Processing

In [6]:
import fitz  # PyMuPDF
import re

def clean_text(text: str) -> str:
    """Clean extracted PDF text."""
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\x00', '', text)
    text = text.replace('ﬁ', 'fi').replace('ﬂ', 'fl')
    return text.strip()

def extract_text_from_pdf(pdf_path: str) -> str:
    """Extract all text from PDF."""
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    doc.close()
    return clean_text(text)

def chunk_text(text: str, chunk_size: int = 1500, overlap: int = 150):
    chunks = []
    start = 0
    chunk_id = 0
    text_length = len(text)

    while start < text_length:
        end = min(start + chunk_size, text_length)

        if end < text_length:
            for punct in ['.', '!', '?']:
                pos = text.rfind(punct, max(start, end - 200), end)
                if pos != -1:
                    end = pos + 1
                    break

        chunk = text[start:end].strip()

        if not chunk:
            break

        chunks.append({
            "chunk_id": chunk_id,
            "text": chunk,
            "start": start,
            "end": end,
            "length": len(chunk),
        })

        chunk_id += 1

        if end >= text_length:
            break

        start = max(end - overlap, start + 1)

    return chunks

print("✅ PDF utilities ready")

✅ PDF utilities ready


In [7]:
# Load and chunk PDF
pdf_path = Path(config['data_root']) / "2024-Annual-Report.pdf"

if not pdf_path.exists():
    raise FileNotFoundError(f"PDF not found: {pdf_path}")

print(f"Extracting text from: {pdf_path.name}")
full_text = extract_text_from_pdf(str(pdf_path))
print(f"✅ Extracted {len(full_text):,} characters")

print(f"\nChunking with size=1500, overlap=150")
chunks = chunk_text(full_text, 1500, 150)
print(f"✅ Created {len(chunks)} chunks")
print(f"   Average length: {sum(c['length'] for c in chunks) / len(chunks):.0f} chars")

Extracting text from: 2024-Annual-Report.pdf
✅ Extracted 620,206 characters

Chunking with size=1500, overlap=150
✅ Created 482 chunks
   Average length: 1436 chars


## Step 2: Q&A Generation

In [9]:
def generate_questions(chunk_text: str, llm, num_questions: int = 10) -> list:
    """Generate exactly 10 questions from a chunk, returning clean JSON."""
    
    prompt = f"""
You are an expert financial analyst creating training data.

Based ONLY on the following text chunk, generate EXACTLY {num_questions} questions.
Cover these categories:
- Hard Facts: specific numbers, dates, names, amounts
- Strategic Summaries: business plans, strategies, future outlook
- Stylistic/Creative: tone of language, implications, wording style

Rules — STRICT:
- Every question must be directly answerable from this chunk alone.
- RETURN ONLY a clean JSON array of {num_questions} strings.
- NO explanations, NO markdown, NO ```json, NO extra text.
- Start directly with [ and end with ] — nothing else.

Chunk:
{chunk_text[:1400]}

JSON array:"""

    try:
        response = llm.invoke(prompt, temperature=0.2)
        text = getattr(response, "content", str(response)).strip()
        print(f"\n🔹 RAW MODEL OUTPUT:\n{text}\n{'-'*80}")

        # Extract JSON array safely
        start = text.find("[")
        end = text.rfind("]") + 1
        if start >= 0 and end > start:
            questions = json.loads(text[start:end])
        else:
            raise ValueError("No JSON array found in model output.")

        if not isinstance(questions, list) or len(questions) != num_questions:
            print(f"⚠️ Warning: Got {len(questions)} questions instead of {num_questions}")
            return []

        return [q.strip() for q in questions]

    except Exception as e:
        print(f"⚠️ Question generation error: {e}")
        return []


def generate_answers(chunk_text: str, questions: list, llm) -> list:
    """LLM B: Generate answers using chunk as context."""
    answers = []
    for q in questions:
        prompt = f"""Answer based ONLY on the context provided. Be precise and concise.

Context:
{chunk_text}

Question: {q}

Answer:"""
        try:
            response = llm.invoke(prompt)
            answers.append(response.content.strip())
        except Exception as e:
            print(f"⚠️  Answer error: {e}")
            answers.append("Error generating answer.")
    return answers

print("✅ Q&A generation functions ready")

✅ Q&A generation functions ready


In [10]:
# Test on sample chunk
test_chunk = chunks[10]['text']
print("Testing Q&A generation...\n")

test_questions = generate_questions(test_chunk, llm, 5)
print(f"✅ Generated {len(test_questions)} questions")
for i, q in enumerate(test_questions, 1):
    print(f"  Q{i}: {q}")

test_answers = generate_answers(test_chunk, test_questions, llm)
print(f"\n✅ Generated {len(test_answers)} answers")
print(f"\nSample Q&A:")
print(f"Q: {test_questions[0]}")
print(f"A: {test_answers[0]}")

Testing Q&A generation...


🔹 RAW MODEL OUTPUT:
[
  "What is the date on which the forward-looking statements in the Annual Report on Form 10-K were made?",
  "What is the name of the company mentioned in the report?",
  "What is the total number of parts in the report's ITEM 1 section?",
  "What is the tone of language used in the warning about relying on forward-looking statements?",
  "What is the name of the report mentioned in the text?"
]
--------------------------------------------------------------------------------
✅ Generated 5 questions
  Q1: What is the date on which the forward-looking statements in the Annual Report on Form 10-K were made?
  Q2: What is the name of the company mentioned in the report?
  Q3: What is the total number of parts in the report's ITEM 1 section?
  Q4: What is the tone of language used in the warning about relying on forward-looking statements?
  Q5: What is the name of the report mentioned in the text?

✅ Generated 5 answers

Sample Q&A:
Q: What

## Step 3: Full Dataset Generation

In [17]:
# Configuration - REDUCED FOR EFFICIENCY
max_chunks = 25
qa_pairs_per_chunk = 10

print(f"🏭 Starting dataset generation...")
print(f"   Chunks: {max_chunks} (out of {len(chunks)} total)")
print(f"   Q&A per chunk: {qa_pairs_per_chunk}")
print(f"   Expected pairs: {max_chunks * qa_pairs_per_chunk}")
print(f"   API calls: ~{max_chunks * 2}")
print(f"   ⏱️ Estimated time: ~{max_chunks * 2} minutes\n")

dataset = []

for chunk in tqdm(chunks[:max_chunks], desc="Generating Q&A"):  # ⭐ ADDED [:max_chunks]
    questions = generate_questions(chunk['text'], llm, qa_pairs_per_chunk)
    if not questions:
        continue
    
    answers = generate_answers(chunk['text'], questions, llm)
    
    for q, a in zip(questions, answers):
        dataset.append({
            'chunk_id': chunk['chunk_id'],
            'question': q,
            'answer': a,
            'context': chunk['text']
        })

print(f"\n✅ Generated {len(dataset)} Q&A pairs")

🏭 Starting dataset generation...
   Chunks: 25 (out of 482 total)
   Q&A per chunk: 10
   Expected pairs: 250
   API calls: ~50
   ⏱️ Estimated time: ~50 minutes



Generating Q&A:   0%|          | 0/25 [00:00<?, ?it/s]


🔹 RAW MODEL OUTPUT:
[
  "What is the fiscal year end date mentioned in the report?",
  "What is the name of the company mentioned in the report?",
  "What is the state of incorporation or organization of Uber Technologies, Inc?",
  "What is the I.R.S. Employer Identification Number of Uber Technologies, Inc?",
  "What is the Commission File Number of Uber Technologies, Inc?",
  "What is the tone of language used in the report?",
  "What is the mission of Uber Technologies, Inc.?",
  "What is the speed mentioned in the report?",
  "What is the date of the report?",
  "What is the name of the regulatory body mentioned in the report?"
]
--------------------------------------------------------------------------------


Generating Q&A:   4%|▍         | 1/25 [00:18<07:26, 18.60s/it]


🔹 RAW MODEL OUTPUT:
[
  "What is the state or other jurisdiction of incorporation or organization of the registrant?",
  "What is the I.R.S. Employer Identification Number of the registrant?",
  "What is the address of the principal executive offices of the registrant?",
  "What is the registrant's telephone number?",
  "What type of security is registered pursuant to Section 12(b) of the Act?",
  "What is the trading symbol(s) of the registered security?",
  "What exchange is the registered security listed on?",
  "What is the par value of the common stock?",
  "Is the registrant a well-known seasoned issuer?",
  "Has the registrant filed all reports required to be filed by Section 13 or 15(d) of the Securities Exchange Act of 1934 during the preceding 12 months?"
]
--------------------------------------------------------------------------------


...


🔹 RAW MODEL OUTPUT:
[
  "What is the name of the European Union's regulation regarding personal data?",
  "When did the GDPR go into effect?",
  "What is the name of the California law that established new consumer rights and data privacy and protection requirements?",
  "When did the CCPA go into effect?",
  "What is the name of the California law passed in 2023 that impacts personal data processing?",
  "When is the California Privacy Rights Act effective?",
  "What is the name of the Indian law enacted in 2023 regarding personal data protection?",
  "How many examples of regulations are mentioned in the text?",
  "What is the core strategy of the company regarding personal data?",
  "In what year did India's Digital Personal Data Protection Act enact?"
]
--------------------------------------------------------------------------------


Generating Q&A:  96%|█████████▌| 24/25 [16:24<00:40, 40.27s/it]


🔹 RAW MODEL OUTPUT:
[
  "What is the name of the California law passed in 2023?",
  "In which year was India's Digital Personal Data Protection Act enacted?",
  "What is the name of Uber's subsidiary in the Netherlands?",
  "What type of institution is Uber Payments B.V. registered as?",
  "In which year did the CPRA become effective?",
  "What is the name of the region where Uber Payments B.V. is authorized to support payment activities?",
  "How many jurisdictions does Uber operate in that have laws governing payment and financial services activities?",
  "What type of licenses does Uber hold in the United Kingdom and Mexico?",
  "What is the potential impact of changes in laws related to money transmission and online payments on Uber's business?",
  "What is the goal of Uber's evaluation of options for seeking further licenses and approvals in several jurisdictions?"
]
--------------------------------------------------------------------------------


Generating Q&A: 100%|██████████| 25/25 [17:02<00:00, 40.88s/it]


✅ Generated 250 Q&A pairs


## Step 4: Split and Save

In [18]:
import random

# Shuffle and split
random.seed(42)
random.shuffle(dataset)

split_idx = int(len(dataset) * 0.8)
train_data = dataset[:split_idx]
test_data = dataset[split_idx:]

print(f"📊 Dataset split:")
print(f"   Train: {len(train_data)} (80%)")
print(f"   Test: {len(test_data)} (20%)")

# Save
artifacts_dir = Path(config['artifacts_root']) / "data_factory"
artifacts_dir.mkdir(parents=True, exist_ok=True)

with open(artifacts_dir / "train.jsonl", 'w') as f:
    for item in train_data:
        f.write(json.dumps(item) + '\n')

with open(artifacts_dir / "golden_test_set.jsonl", 'w') as f:
    for item in test_data:
        f.write(json.dumps(item) + '\n')

print(f"\n✅ Saved to {artifacts_dir}")
print(f"   - train.jsonl")
print(f"   - golden_test_set.jsonl")
print(f"\n🎉 Data Factory Complete!")

📊 Dataset split:
   Train: 200 (80%)
   Test: 50 (20%)

✅ Saved to artifacts/data_factory
   - train.jsonl
   - golden_test_set.jsonl

🎉 Data Factory Complete!
